In [ ]:
!pip install -U "torchao>=0.16.0"
!pip install -q -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 77.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 107.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 33.4 MB/s eta 0:00:00


In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import time
import torch
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

TRAIN_PATH = "/kaggle/input/competitions/dlp-nppe-1-t-22026/train.csv"
TEST_PATH = "/kaggle/input/competitions/dlp-nppe-1-t-22026/test.csv"

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


cuda


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

unique_labels = sorted(train_df["label"].unique())

label2id_int = {int(l): i for i, l in enumerate(unique_labels)}
label2id = {str(int(l)): i for i, l in enumerate(unique_labels)}
id2label = {i: str(int(l)) for i, l in enumerate(unique_labels)}

train_df["label_id"] = train_df["label"].map(label2id_int).astype(int)
num_labels = len(unique_labels)
print(f"Loaded {len(train_df)} training rows. Found {num_labels} distinct categories.")

from huggingface_hub import login
HF_TOKEN = "" #Token removed for form submission
login(token=HF_TOKEN)

Loaded 50840 training rows. Found 30 distinct categories.


In [ ]:
def build_datasets(tokenizer, max_length):
    def tok_fn(examples):
        return tokenizer(examples["text"], truncation=True, max_length=max_length)

    hf_train = Dataset.from_pandas(train_df[["text", "label_id"]].rename(columns={"label_id": "labels"}))
    hf_test = Dataset.from_pandas(test_df[["text"]])

    tok_train = hf_train.map(tok_fn, batched=True, remove_columns=["text"])
    tok_test = hf_test.map(tok_fn, batched=True, remove_columns=["text"])
    return tok_train, tok_test


def train_full_finetune(model_name, output_dir, seed, epochs=2, lr=2e-5,
                         batch_size=16, grad_accum=1, max_length=512, model_kwargs=None):
    model_kwargs = model_kwargs or {}
    set_seed(seed)
    stage_start = time.time()
    print(f"\n===== [{time.strftime('%H:%M:%S')}] FULL fine-tune {model_name} seed={seed} "
          f"(epochs={epochs}, max_length={max_length}, batch={batch_size}, grad_accum={grad_accum}) =====")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tok_train, tok_test = build_datasets(tokenizer, max_length=max_length)
    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=num_labels, id2label=id2label, label2id=label2id, **model_kwargs
    )
    model.to(device)

    training_args = TrainingArguments(
        output_dir=f"{output_dir}_seed{seed}",
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        num_train_epochs=epochs,
        weight_decay=0.01,
        warmup_ratio=0.06,
        logging_steps=50,
        logging_first_step=True,
        disable_tqdm=False,
        save_strategy="no",
        fp16=torch.cuda.is_available(),
        report_to="none",
        seed=seed,
    )

    trainer = Trainer(model=model, args=training_args, train_dataset=tok_train, data_collator=collator)
    trainer.train()

    elapsed = time.time() - stage_start
    print(f"===== [{time.strftime('%H:%M:%S')}] Finished {model_name} seed={seed} "
          f"in {elapsed/60:.1f} min =====")

    preds = trainer.predict(tok_test)
    logits = torch.tensor(preds.predictions)
    probs = torch.softmax(logits, dim=1).numpy()

    del model, trainer
    torch.cuda.empty_cache()
    return probs

In [ ]:
SEEDS = [42, 1234, 2026]

legalbert_probs_list = []
for s in SEEDS:
    probs = train_full_finetune(
        model_name="nlpaueb/legal-bert-base-uncased",
        output_dir="./legal_bert_full_nppe",
        seed=s,
        epochs=4, #3,
        lr= 5e-5, #2e-5,
        batch_size=8, #16,
        grad_accum=2, #1,
        max_length=512,
    )
    legalbert_probs_list.append(probs)

legalbert_ensemble_probs = np.mean(legalbert_probs_list, axis=0)

predicted_class_ids = np.argmax(legalbert_ensemble_probs, axis=1)
predicted_labels = [id2label[c] for c in predicted_class_ids]


===== [08:34:18] FULL fine-tune nlpaueb/legal-bert-base-uncased seed=42 (epochs=4, max_length=512, batch=8, grad_accum=2) =====


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

[transformers] You passed `num_labels=30` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 

Step,Training Loss
1,13.979017
50,13.793112
100,12.284266
150,10.073604
200,8.046744
250,6.893370
300,6.055259
350,5.891215
400,5.481769
450,5.244011


===== [18:48:42] Finished nlpaueb/legal-bert-base-uncased seed=2026 in 202.3 min =====


In [ ]:
submission = pd.DataFrame({
    "ID": test_df["id"],
    "label": predicted_labels,
})
submission.to_csv("/kaggle/working/submission.csv", index=False)
print("Saved submission.csv")
submission.head()

Saved submission.csv


,ID,label
0,0,7
1,1,2
2,2,3
3,3,11
4,4,0
